# Does `generate_content` mean one LLM call, or a loop?

Short answer: **it depends on what's in `tools`, and on Automatic Function Calling (AFC).**

- No tools, or a *server-side* tool (Google Search, code execution): exactly **one** network round trip. Google's backend does any searching/code-running internally, inside that one call — the SDK is a passive pipe.
- A **Python callable** in `tools=[...]`, with AFC on (the default since google-genai 2.8.0, capped at 10 hops): the SDK itself loops — **multiple** `generate_content` round trips, with your Python function actually invoked in between — all inside what looks like one line of your code.
- The same Python tool, with AFC explicitly disabled: back to exactly **one** round trip — but now you get a raw, *unexecuted* function-call part back, and have to run the loop yourself.

This notebook proves each case empirically rather than just asserting it — with a call counter on the tool function and by inspecting `response.automatic_function_calling_history`.

**Setup** — assumes `.env` is configured and you've run `gcloud auth application-default login`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join('..', 'samples')))
from common import DEFAULT_MODEL, get_client
from google.genai import types

# A call log so we can SEE, from outside, how many times the SDK
# actually invoked our Python function — proof, not just a claim.
call_log: list[str] = []

def get_weather(city: str) -> dict:
    """Get the current weather for a city.

    Args:
        city: the city name, e.g. "Boston".
    """
    call_log.append(city)
    return {'city': city, 'condition': 'sunny', 'temp_c': 22}


def describe_history(history):
    """Label each entry: a real LLM generation (role == 'model') vs.
    an input fed to the model (role == 'user') — not the same thing,
    even though both are entries in the same list."""
    generations = 0
    for i, turn in enumerate(history, start=1):
        is_generation = turn.role == 'model'
        generations += is_generation
        kind = 'GENERATION (model output)' if is_generation else 'input (fed to the model)'
        for part in turn.parts:
            if part.text:
                detail = f'text: {part.text!r}'
            elif part.function_call:
                detail = f'function_call: {part.function_call.name}({dict(part.function_call.args)})'
            elif part.function_response:
                detail = f'function_response: {part.function_response.name} -> {part.function_response.response}'
            else:
                detail = str(part)
            print(f'  [{i}] {turn.role:<6} {kind:<28} {detail}')
    return generations


## Case A — no tools

The baseline: always exactly one round trip. `automatic_function_calling_history` is `None` because AFC never engages when there's nothing to call.

In [ ]:
client = get_client()
response = client.models.generate_content(
    model=DEFAULT_MODEL,
    contents='In one sentence, what is a vector database?',
)
print('text:', response.text)
print('automatic_function_calling_history:', response.automatic_function_calling_history)


## Case B — a server-side tool (Google Search)

Still exactly **one** round trip from the client's point of view. Google's backend does the searching internally, inside that one call — you never see an intermediate step, because there isn't one on your end. `automatic_function_calling_history` stays empty here too: AFC is a client-side mechanism for *your* Python functions, not for Google's own built-in tools.

In [ ]:
response = client.models.generate_content(
    model=DEFAULT_MODEL,
    contents="What's one notable AI news item from this week?",
    config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())]),
)
print('text:', response.text[:200], '...')
print('automatic_function_calling_history:', response.automatic_function_calling_history)


## Case C — a Python tool, AFC ON (the default)

One call *from your code's perspective*. But watch `call_log` and `automatic_function_calling_history` after running this cell: the SDK made more than one model request, and it called `get_weather()` itself — you never wrote `get_weather(...)` anywhere in this cell.

**Important caveat**: this is shown for the mechanics, not as the recommended pattern. `google-genai` itself warns that using AFC through `generate_content` directly is deprecated — a future major version removes AFC support from `generate_content`/`generate_content_stream` entirely, making `Chat.send_message()` the only supported way to use it. That's exactly why every *other* agent sample in this repo (`03`, `11`, `13`, `14`) already uses `chats.create()` instead of calling `generate_content` directly.

In [ ]:
call_log.clear()
response = client.models.generate_content(
    model=DEFAULT_MODEL,
    contents="What's the weather in Boston?",
    config=types.GenerateContentConfig(tools=[get_weather]),
)
print('text:', response.text)
print('our function was called', len(call_log), 'time(s); it logged:', call_log)

history = response.automatic_function_calling_history or []
print(f'\nautomatic_function_calling_history has {len(history)} entries:')
generations = describe_history(history)
print(f'-> {generations} of those are actual model generations; the rest are inputs fed to the model.')


## Case D — same Python tool, AFC explicitly OFF

Now `generate_content` really is just one call again — it hands back an **unexecuted** `function_call` part instead of running anything. We then do, by hand, exactly what AFC was doing invisibly in Case C: execute the function ourselves, and call `generate_content` a second time with the result attached.

In [ ]:
call_log.clear()
response = client.models.generate_content(
    model=DEFAULT_MODEL,
    contents="What's the weather in Boston?",
    config=types.GenerateContentConfig(
        tools=[get_weather],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    ),
)

call_part = response.candidates[0].content.parts[0].function_call
print('model asked to call:', call_part.name, dict(call_part.args))
print('our function was called', len(call_log), "time(s) so far — the SDK did NOT run it.")


In [ ]:
# Step 2: WE execute the tool ourselves, since the SDK won't.
result = get_weather(**call_part.args)
print('manually executed get_weather ->', result)

# Step 3: WE build the follow-up turn and call generate_content AGAIN,
# ourselves — this is exactly the round trip AFC was doing invisibly above.
follow_up_contents = [
    types.Content(role='user', parts=[types.Part.from_text(text="What's the weather in Boston?")]),
    response.candidates[0].content,  # the model's own function_call turn
    types.Content(
        role='user',
        parts=[types.Part.from_function_response(name=call_part.name, response=result)],
    ),
]
response2 = client.models.generate_content(
    model=DEFAULT_MODEL,
    contents=follow_up_contents,
    config=types.GenerateContentConfig(tools=[get_weather]),
)
print('final text, from OUR second call:', response2.text)
print('our function was called', len(call_log), 'time(s) total — exactly once, by us.')


## Case E — counting the actual network calls directly

Two ways to see "each set of generations" exist, at different levels:

- **Semantic**: `response.automatic_function_calling_history` above already *is* each generation, as real Content objects — but it only exists for the AFC case (C).
- **Universal**: hook `httpx.Client.send` directly. `google-genai` uses `httpx` internally, and deliberately suppresses `httpx`'s own DEBUG logging (to avoid leaking API keys/payloads) — so bumping `logging` to DEBUG won't show much. Hooking `send()` itself sits beneath that suppression and can't be silenced; it counts every real network call no matter which case is running.

**Limitation to be honest about**: this proves the *client-visible* call count precisely. It still cannot see inside a single HTTP response — so for Case B, it still can't tell us how many times Google's backend invoked the model to produce that one response. That part stays unobservable from here, no matter what we hook.

In [ ]:
import contextlib
import httpx

@contextlib.contextmanager
def count_http_calls():
    count = 0
    original_send = httpx.Client.send

    def counting_send(self, request, *args, **kwargs):
        nonlocal count
        count += 1
        print(f'  [HTTP call #{count}] {request.method} {request.url.path}')
        return original_send(self, request, *args, **kwargs)

    httpx.Client.send = counting_send
    try:
        yield lambda: count
    finally:
        httpx.Client.send = original_send


Re-running each case wrapped in the counter — watch the printed call count match (or not!) what we predicted above.

In [ ]:
print('--- Case A ---')
with count_http_calls() as get_count:
    response = client.models.generate_content(model=DEFAULT_MODEL, contents='Hi')
print('total HTTP calls:', get_count())


In [ ]:
print('--- Case B ---')
with count_http_calls() as get_count:
    response = client.models.generate_content(
        model=DEFAULT_MODEL,
        contents="What's one notable AI news item from this week?",
        config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())]),
    )
print('total HTTP calls:', get_count())


In [ ]:
print('--- Case C ---')
call_log.clear()
with count_http_calls() as get_count:
    response = client.models.generate_content(
        model=DEFAULT_MODEL,
        contents="What's the weather in Boston?",
        config=types.GenerateContentConfig(tools=[get_weather]),
    )
print('total HTTP calls:', get_count(), '— compare to len(call_log) =', len(call_log))


## Takeaway

| | Tool type | AFC | Round trips per Python call | Who runs the function |
|---|---|---|---|---|
| A | none | — | 1 | nobody |
| B | server-side (Search/code exec) | — | 1 (looks like a loop, but it's inside Google's backend) | Google's backend |
| C | Python callable | on (default) | 2+ (one per tool-call hop, capped at 10) | the SDK, automatically |
| D | Python callable | off | 1 per call you make | you, manually |

`client.chats.create()` / `chat.send_message()` doesn't change any of this — it's the same four cases, just with the SDK also remembering prior turns for you.

Case E's `count_http_calls()` is the general-purpose way to verify any of this yourself, for any case, without relying on a response field that might not exist.